# 1.DataSet Processing

In [1]:
import json
import os
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
nltk.download('punkt')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
	"""Preprocesses text by lowercasing, tokenizing, removing stopwords and stemming."""
	tokens = word_tokenize(text.lower())
	filtered_tokens = [stemmer.stem(word) for word in tokens if word.isalnum() and word not in stop_words]
	filtered_tokens = [stemmer.stem(word) for word in filtered_tokens if not word.isnumeric()]
	return ' '.join(filtered_tokens)
	
# Load JSON data
def load_data(filepath):
	with open(filepath, 'r') as file:
		data = json.load(file)
	return data

[nltk_data] Downloading package punkt to /Users/chenluyao/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [40]:
# evidence_data = load_data('data/evidence.json')
# evidence_map = {eid: preprocess_text(text) for eid, text in evidence_data.items()}


# def save_to_json(filepath, data):
#     """ Save a dictionary to a JSON file. """
#     with open(filepath, 'w', encoding='utf-8') as f:
#         json.dump(data, f, ensure_ascii=False, indent=4)

# # Example usage
# save_to_json('data/curated/preprocessed_evidence_map.json', evidence_map)

In [41]:
train_claims_data = load_data('data/train-claims.json')
evidence_data = load_data('data/evidence.json')
dev_claims_data = load_data('data/dev-claims.json')
evidence_map = load_data('data/curated/preprocessed_evidence_map.json')
filtered_evidence_map = load_data('filtered_evidence_map.json')
test_claims_data = load_data('data/test-claims-unlabelled.json')

In [5]:
# evidence_df = pd.DataFrame(evidence_map.items(), columns=['id', 'evidence'])
# evidence_df

In [6]:
filtered_evidence_df = pd.DataFrame(filtered_evidence_map.items(), columns=['id', 'evidence'])
filtered_evidence_df

,id,evidence
0,evidence-0,john bennet law english entrepreneur and agric...
1,evidence-1,lindberg began hi profession career at the age...
2,evidence-3,gerald franci goyer born octob wa a profession...
3,evidence-7,in addit to known and tangibl risk unforese bl...
4,evidence-12,he is the current aida individu world champion...
...,...,...
298683,evidence-895097,snowflak fell on 19 out of 28 day in the bosto...
298684,evidence-906284,the long australian millenni drought broke in ...
298685,evidence-1025757,in addit mani area are experienc higher than n...
298686,evidence-403673,global warm of 1.5


In [7]:
data_for_dataframe = []
for claim_id, claim_details in train_claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'])
    claim_label = claim_details['claim_label']
    eids = claim_details['evidences']
    data_for_dataframe.append({
			'claim_id': claim_id,
            'claim': claim_text,
            'claim_text_raw': claim_details['claim_text'],
            'evidence': eids,
            'label': claim_label
        })

# Create DataFrame
train_claims_df = pd.DataFrame(data_for_dataframe)

train_claims_df['evidence_texts'] = train_claims_df['evidence'].apply(
    lambda x: [filtered_evidence_map[evidence_id] for evidence_id in x]
)

train_claims_df

,claim_id,claim,claim_text_raw,evidence,label,evidence_texts
0,claim-1937,scientif evid co2 pollut higher co2 concentr a...,Not only is there no scientific evidence that ...,"[evidence-442946, evidence-1194317, evidence-1...",DISPUTED,[at veri high concentr 100 time atmospher conc...
1,claim-126,el niño drove record high global temperatur su...,El Niño drove record highs in global temperatu...,"[evidence-338219, evidence-1127398]",REFUTES,[while climat chang can be due to natur forc o...
2,claim-2510,pdo switch cool phase,"In 1946, PDO switched to a cool phase.","[evidence-530063, evidence-984887]",SUPPORTS,[there is evid of revers in the prevail polar ...
3,claim-2021,weather channel john coleman provid evid convi...,Weather Channel co-founder John Coleman provid...,"[evidence-1177431, evidence-782448, evidence-5...",DISPUTED,[there is no convinc scientif evid that human ...
4,claim-2449,januari cap month period global temperatur dro...,"""January 2008 capped a 12 month period of glob...","[evidence-1010750, evidence-91661, evidence-72...",NOT_ENOUGH_INFO,"[with averag temperatur +8.1 47, the iranian p..."
...,...,...,...,...,...,...
1223,claim-1504,climat scientist say aspect case hurrican harv...,Climate scientists say that aspects of the cas...,"[evidence-1055682, evidence-1047356, evidence-...",SUPPORTS,[it a fact climat chang made hurrican harvey m...
1224,claim-243,5th assess report ipcc estim human emiss proba...,"In its 5th assessment report in 2013, the IPCC...",[evidence-916755],SUPPORTS,[the scientif consensu as of 2013 updat as sta...
1225,claim-2302,sinc mid global temperatur warm around degr ce...,"Since the mid 1970s, global temperatures have ...","[evidence-403673, evidence-889933, evidence-11...",NOT_ENOUGH_INFO,"[global warm of 1.5, multipl independ produc i..."
1226,claim-502,abnorm temperatur spike februari earlier month...,But abnormal temperature spikes in February an...,"[evidence-97375, evidence-562427, evidence-521...",NOT_ENOUGH_INFO,[a lower air temperatur of wa record in 2010 b...


In [35]:
data_for_dataframe = []
for claim_id, claim_details in dev_claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'])
    claim_label = claim_details['claim_label']
    eids = claim_details['evidences']
    data_for_dataframe.append({
            'claim_id': claim_id,
            'claim': claim_text,
            'claim_text_raw': claim_details['claim_text'],
        })

# Create DataFrame
dev_claims_df = pd.DataFrame(data_for_dataframe)
dev_claims_df

,claim_id,claim,claim_text_raw
0,claim-752,south australia expen electr world,[South Australia] has the most expensive elect...
1,claim-375,per cent total annual global emiss carbon diox...,when 3 per cent of total annual global emissio...
2,claim-1266,mean world 1c warmer time,This means that the world is now 1C warmer tha...
3,claim-871,happen zika may also good model second worri e...,"“As it happens, Zika may also be a good model ..."
4,claim-2164,greenland lost tini fraction ice mass,Greenland has only lost a tiny fraction of its...
...,...,...,...
149,claim-2400,suddenli label co2 pollut disserv ga play enor...,"'To suddenly label CO2 as a ""pollutant"" is a d..."
150,claim-204,natur orbit driven warm atmosph carbon dioxid ...,"after a natural orbitally driven warming, atmo..."
151,claim-1426,mani world coral reef alreadi barren state con...,Many of the world’s coral reefs are already ba...
152,claim-698,recent studi led lawrenc livermor nation labor...,A recent study led by Lawrence Livermore Natio...


In [28]:
data_for_dataframe = []
for claim_id, claim_details in test_claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'])
    data_for_dataframe.append({
			'claim_id': claim_id,
            'claim': claim_text,
            'claim_text_raw': claim_details['claim_text']
        })
    
# Create DataFrame
test_claims_df = pd.DataFrame(data_for_dataframe)
test_claims_df 

,claim_id,claim,claim_text_raw
0,claim-2967,contribut wast heat global climat,The contribution of waste heat to the global c...
1,claim-979,warm weather worsen recent drought includ drie...,“Warm weather worsened the most recent five-ye...
2,claim-1609,greenland lost tini fraction ice mass,Greenland has only lost a tiny fraction of its...
3,claim-1020,global reef crisi necessarili mean extinct cor...,“The global reef crisis does not necessarily m...
4,claim-2599,small amount activ substanc cau larg effect,Small amounts of very active substances can ca...
...,...,...,...
148,claim-293,measur equip get old need replac often requir,When the measuring equipment gets old and need...
149,claim-910,cement iron steel petroleum refin industri cou...,"The cement, iron and steel, and petroleum refi..."
150,claim-2815,new studi surfac warm solar cycl found time hi...,A new peer-reviewed study on Surface Warming a...
151,claim-1652,strong co2 effect observ mani differ measur,The strong CO2 effect has been observed by man...


In [9]:
all_texts = train_claims_df['claim'].tolist() + filtered_evidence_df['evidence'].tolist()

# Vectorization
vectorizer = TfidfVectorizer(max_features=1000)

# Fit the vectorizer on both claims and evidences
vectorizer.fit(all_texts) 
claim_vec = vectorizer.transform(train_claims_df['claim']).toarray() 
train_claims_df['claim_tfidf'] = list(claim_vec) 
claim_vec.shape

(1228, 1000)

In [10]:
filtered_evidence_vec = vectorizer.transform(filtered_evidence_df['evidence']).toarray()
filtered_evidence_df['evidence_tfidf'] = list(filtered_evidence_vec) 
filtered_evidence_vec.shape

(298688, 1000)

In [11]:
dev_claim_vec = vectorizer.transform(dev_claims_df['claim']).toarray() 
dev_claims_df['claim_tfidf'] = list(dev_claim_vec) 
dev_claim_vec.shape

(154, 1000)

In [29]:
test_claim_vec = vectorizer.transform(test_claims_df['claim']).toarray() 
test_claims_df['claim_tfidf'] = list(test_claim_vec) 
test_claim_vec.shape

(153, 1000)

In [30]:
def top_k_evidence(claim_df, evidence_df, evidence_map, k=5):
	# compute cosine similarity between each claim and each evidence
	X = np.array(claim_df['claim_tfidf'].values.tolist())
	y = np.array(evidence_df['evidence_tfidf'].values.tolist())
	sim = cosine_similarity(X, y)

	# get top k evidence with highest similarity score with the claim
	data = np.zeros((sim.shape[0], k))
	top_evidence_id = []
	for i in range(sim.shape[0]):
		data[i] = np.argpartition(sim[i], -k)[-k:]
		top_evidence_id.append([evidence_df.iloc[int(ind)]['id'] for ind in data[i]])

	claim_df['top5_evidence_id'] = top_evidence_id

	claim_df = claim_df[["claim_id", "claim_text_raw", "top5_evidence_id"]]

	# get texts of top k evidence
	claim_df['evidence_texts'] = claim_df['top5_evidence_id'].apply(
		lambda x: [evidence_map[evidence_id] for evidence_id in x]
	)
	return claim_df


In [ ]:
dev_claims_df = top_k_evidence(dev_claims_df, filtered_evidence_df, filtered_evidence_map)
dev_claims_df.to_csv("data/curated/dev_evidence_retrieval.csv", index=False)

In [31]:
test_claims_df = top_k_evidence(test_claims_df, filtered_evidence_df, filtered_evidence_map)
test_claims_df.to_csv("data/curated/test_evidence_retrieval2.csv", index=False)
test_claims_df

/var/folders/df/4qk5nt6555bggnc39502n5b80000gn/T/ipykernel_37917/2447078132.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  claim_df['evidence_texts'] = claim_df['top5_evidence_id'].apply(


,claim_id,claim_text_raw,top5_evidence_id,evidence_texts
0,claim-2967,The contribution of waste heat to the global c...,"[evidence-1090341, evidence-621274, evidence-1...",[global climat forum a platform for joint stud...
1,claim-979,“Warm weather worsened the most recent five-ye...,"[evidence-417353, evidence-1119884, evidence-8...",[climat is the statist usual mean or variabl o...
2,claim-1609,Greenland has only lost a tiny fraction of its...,"[evidence-891137, evidence-487116, evidence-11...",[the gradual accumul of ice on the laurentid i...
3,claim-1020,“The global reef crisis does not necessarily m...,"[evidence-151390, evidence-1152843, evidence-7...","[myllyoja liter mean millditch, mean absolut p..."
4,claim-2599,Small amounts of very active substances can ca...,"[evidence-347797, evidence-1004410, evidence-7...",[the wolff chaikoff effect is an effect mean o...
...,...,...,...,...
148,claim-293,When the measuring equipment gets old and need...,"[evidence-751374, evidence-104953, evidence-48...",[the new measur requir climat model paramet ad...
149,claim-910,"The cement, iron and steel, and petroleum refi...","[evidence-114183, evidence-42541, evidence-379...",[the theme of hi work is to live a happi prosp...
150,claim-2815,A new peer-reviewed study on Surface Warming a...,"[evidence-500457, evidence-694951, evidence-18...",[the solar storm of known as the carrington ev...
151,claim-1652,The strong CO2 effect has been observed by man...,"[evidence-429904, evidence-16131, evidence-229...",[clinomet measur both inclin posit slope as se...


### Claim Classification

In [42]:
data_for_dataframe = []
for claim_id, claim_details in dev_claims_data.items():
    claim_text = claim_details['claim_text']
    claim_label = claim_details['claim_label']
    eids = claim_details['evidences']
    data_for_dataframe.append({
            'claim': claim_text,
            'evidence': eids,
            'label': claim_label
        })

# Create DataFrame
dev_claims_df = pd.DataFrame(data_for_dataframe)

dev_claims_df['evidence_texts'] = dev_claims_df['evidence'].apply(
    lambda x: [evidence_map[evidence_id] for evidence_id in x]
)

dev_claims_df

,claim,evidence,label,evidence_texts
0,[South Australia] has the most expensive elect...,"[evidence-67732, evidence-572512]",SUPPORTS,[citat need south australia highest retail pri...
1,when 3 per cent of total annual global emissio...,"[evidence-996421, evidence-1080858, evidence-2...",NOT_ENOUGH_INFO,[unep green economi report state agricultur op...
2,This means that the world is now 1C warmer tha...,"[evidence-889933, evidence-694262]",SUPPORTS,[multipl independ produc instrument dataset co...
3,"“As it happens, Zika may also be a good model ...","[evidence-422399, evidence-702226, evidence-28...",NOT_ENOUGH_INFO,[genet disord result deleteri mutat due sponta...
4,Greenland has only lost a tiny fraction of its...,"[evidence-52981, evidence-264761, evidence-947...",REFUTES,[iceberg calv happen averag greenland lost gt ...
...,...,...,...,...
149,"'To suddenly label CO2 as a ""pollutant"" is a d...","[evidence-409365, evidence-127519, evidence-85...",REFUTES,[state articl convent requir greenhou ga ghg c...
150,"after a natural orbitally driven warming, atmo...","[evidence-368192, evidence-261690, evidence-20...",NOT_ENOUGH_INFO,[increa atmosph concentr co greenhou gase meth...
151,Many of the world’s coral reefs are already ba...,"[evidence-1124018, evidence-995813, evidence-1...",NOT_ENOUGH_INFO,[tropic water contain nutrient yet coral reef ...
152,A recent study led by Lawrence Livermore Natio...,[evidence-660755],REFUTES,[studi david douglass cowork conclud commonli ...


In [43]:
# combine claim text and evidence texts
X_train = train_claims_df['claim'] + train_claims_df['evidence_texts'].apply(lambda x: ' '.join(x))
y_train = train_claims_df['label']

X_dev = dev_claims_df['claim'] + dev_claims_df['evidence_texts'].apply(lambda x: ' '.join(x))
y_dev = dev_claims_df['label']

X_test = test_claims_df['claim_text_raw'] + test_claims_df['evidence_texts'].apply(lambda x: ' '.join(x))

count_vectorizer = CountVectorizer()
X_train_count = count_vectorizer.fit_transform(X_train)
X_dev_count = count_vectorizer.transform(X_dev)
X_test_count = count_vectorizer.transform(X_test)

In [46]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Hyperparameters
n_estimators_values = [50, 100, 200]
max_depth_values = [None, 10, 20]

accuracy_scores_rf = []
for n_estimators in n_estimators_values:
    for max_depth in max_depth_values:
        rf_classifier = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        rf_classifier.fit(X_train_count, y_train)
        y_pred_rf = rf_classifier.predict(X_dev_count)
        
        accuracy_rf = accuracy_score(y_dev, y_pred_rf)
        accuracy_scores_rf.append(((n_estimators, max_depth), accuracy_rf))
        print(f"n_estimators = {n_estimators}, max_depth = {max_depth}: Accuracy = {accuracy_rf}")

print("Accuracy scores for Random Forest:")
for params, accuracy in accuracy_scores_rf:
    print(f"Parameters: {params}, Accuracy: {accuracy}")

n_estimators = 50, max_depth = None: Accuracy = 0.43506493506493504
n_estimators = 50, max_depth = 10: Accuracy = 0.44155844155844154
n_estimators = 50, max_depth = 20: Accuracy = 0.44805194805194803
n_estimators = 100, max_depth = None: Accuracy = 0.42857142857142855
n_estimators = 100, max_depth = 10: Accuracy = 0.44805194805194803
n_estimators = 100, max_depth = 20: Accuracy = 0.44155844155844154
n_estimators = 200, max_depth = None: Accuracy = 0.4155844155844156
n_estimators = 200, max_depth = 10: Accuracy = 0.44155844155844154
n_estimators = 200, max_depth = 20: Accuracy = 0.44155844155844154
Accuracy scores for Random Forest:
Parameters: (50, None), Accuracy: 0.43506493506493504
Parameters: (50, 10), Accuracy: 0.44155844155844154
Parameters: (50, 20), Accuracy: 0.44805194805194803
Parameters: (100, None), Accuracy: 0.42857142857142855
Parameters: (100, 10), Accuracy: 0.44805194805194803
Parameters: (100, 20), Accuracy: 0.44155844155844154
Parameters: (200, None), Accuracy: 0.4155

In [ ]:
# Random Forest 
rf_classifier = RandomForestClassifier(n_estimators=50, max_depth=20, random_state=42)
rf_classifier.fit(X_train_count, y_train)
y_pred = rf_classifier.predict(X_test_count)
test_claims_df["label"] = y_pred
test_claims_df['evidences'] = test_claims_df['top5_evidence_id'].apply(
    lambda x: ["evidence-" + str(evidence_id) for evidence_id in x]
)
test_claims_df.drop(columns=['evidence_texts', 'top5_evidence_id'], inplace=True)
test_claims_df.rename(columns={"claim_text_raw": "claim_text", "label": "claim_label"}, inplace=True)
test_claims_df.set_index('claim_id', inplace=True)
test_claims_df

In [48]:
# convert to json file
from json import loads
result = test_claims_df.to_json(orient="index")
with open('data/curated/test-output2.json', 'w') as f:
    f.write(result)